In [1]:
import json
import os

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.asyncio import tqdm

from internal.data.mil_dataset import MILDatasetMemmapRanges
from internal.data.shuffle_and_cap_bag import ShuffleAndCapBag
from internal.nn.attention_mil import AttentionMIL

cuda_is_available = torch.cuda.is_available()

config = json.load(
    open(os.path.join("processed", "config.json"), "r")
)
OUT_DIR = config["OUT_DIR"]
X_PATH = config["X_PATH"]
M_PATH = config["M_PATH"]
Y_PATH = config["Y_PATH"]
IDX_PATH = config["IDX_PATH"]
LOG_PATH = config["LOG_PATH"]
PATCH_SIZE = config["PATCH_SIZE"]
N_PATCHES = config["N_PATCHES"]
MARGIN = config["MARGIN"]
MASK_PATCH_FRAC = config["MASK_PATCH_FRAC"]
MIN_MASK_PIXELS_SLIDE = config["MIN_MASK_PIXELS_SLIDE"]
MIN_MASK_IN_PATCH = config["MIN_MASK_IN_PATCH"]
MIN_TISSUE_FRAC = config["MIN_TISSUE_FRAC"]
MIN_PATCH_PER_SLIDE = config["MIN_PATCH_PER_SLIDE"]
MIN_CENTER_DIST = config["MIN_CENTER_DIST"]
MAX_TRIES_PER_SLIDE = config["MAX_TRIES_PER_SLIDE"]
MAX_TRIES_PER_PATCH = config["MAX_TRIES_PER_PATCH"]
CLASS2ID = config["CLASS2ID"]
ID2CLASS = config["ID2CLASS"]

RETURN_META = True

In [2]:
def mil_collate(batch):
    # batch items are (x, y) or (x, y, meta)
    xs = [b[0] for b in batch]
    ys = torch.stack([b[1] if torch.is_tensor(b[1]) else torch.tensor(b[1], dtype=torch.long) for b in batch]).long()

    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)  # (sum n_i, C, H, W)

    return xcat, ys, bag_sizes

def mil_collate_with_meta(batch):
    xs = [b[0] for b in batch]
    ys = torch.stack([b[1] if torch.is_tensor(b[1]) else torch.tensor(b[1], dtype=torch.long) for b in batch]).long()
    metas = [b[2] for b in batch]  # list of dicts

    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)

    return xcat, ys, bag_sizes, metas

def mil_collate_concat(batch):
    if RETURN_META:
        xs, ys, _ = zip(*batch)   # each x: (n_i,C,H,W)
    else:
        xs, ys = zip(*batch)      # each x: (n_i,C,H,W)
    bag_sizes = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    x = torch.cat(xs, dim=0)  # (sum n_i, C,H,W)
    y = torch.stack(ys)       # (B,)
    return x, y, bag_sizes

In [4]:
train_ds = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)
val_ds   = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)
train_loader = DataLoader(
    train_ds,
    batch_size=4,              # number of slides per batch
    shuffle=True,
    num_workers=4,
    pin_memory=cuda_is_available,
    collate_fn=mil_collate_concat,   # or mil_collate_list
)

# Sanity: one batch
rgb, msk, label = train_ds[0]
print("Train dataset sample (rgb, msk, label):")
print(rgb.shape, msk.shape, label)

rgb, msk, label = val_ds[0]
print("Val dataset sample (rgb, msk, label):")
print(rgb.shape, msk.shape, label)

x, y, bag_sizes = next(iter(train_loader))
print("Train loader batch (x, y, bag_sizes):")
print(x.shape, y.shape, bag_sizes.shape)

Train dataset sample (rgb, msk, label):
torch.Size([9, 4, 384, 384]) torch.Size([]) {'slide_index': 0, 'start': 0, 'end': 9, 'bag_size': 9}
Val dataset sample (rgb, msk, label):
torch.Size([9, 4, 384, 384]) torch.Size([]) {'slide_index': 0, 'start': 0, 'end': 9, 'bag_size': 9}
Train loader batch (x, y, bag_sizes):
torch.Size([37, 4, 384, 384]) torch.Size([4]) torch.Size([4])


In [ ]:
# take 4 slides only
small_ds = torch.utils.data.Subset(train_ds, [0,1,2,3])
small_loader = DataLoader(
    small_ds,
    batch_size=4,
    collate_fn=mil_collate,
    shuffle=True
)

model = AttentionMIL(n_classes=4).cuda() if cuda_is_available else AttentionMIL(n_classes=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for step in tqdm(range(200), desc="training health check"):
    xcat, y, bag_sizes = next(iter(small_loader))
    if cuda_is_available:
        xcat, y, bag_sizes = xcat.cuda(), y.cuda(), bag_sizes.cuda()

    logits = model(xcat, bag_sizes)
    loss = loss_fn(logits, y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 20 == 0:
        print(step, loss.item())
